# Answer assessments and paired outcomes

This notebook reads fixed assessments, checks their applicability and forms
question-matched contrasts. It keeps technical missingness, semantic non-applicability
and valid zero scores separate. Default inputs are explicitly illustrative values,
with no claimed human rating of the saved RFC model answers.

Use the main environment with the `evaluation` extra described in the README.
Accepted private inputs use the same manifest and functions in an ignored executed
copy. See thesis Sections 2.6–2.7 and Appendix A.1–A.2 for the definitions.

In [1]:
import hashlib
import json
import os
from pathlib import Path

from IPython.display import Markdown, display
from tabulate import tabulate

from rag_comparison import config
from rag_comparison.artifacts import read_jsonl, verify_artifacts
from rag_comparison.assessments import answer_outcomes, behavior_counts
from rag_comparison.evaluation import (
    connected_components,
    paired_intervention_change,
    paired_system_gap_shift,
    surface_coverage,
)
from rag_comparison.evaluation import outcome_inputs
from rag_comparison.artifacts import write_analysis_manifest

## Bind the evaluation scope and assessment revision

PRIMARY and FINAL retain different shared relevance pools. The selected manifest
also identifies the assessment revision and acceptance status. Genuine User keeps
its own revision and nominal categories in the next notebook. Changed model responses
require their own assessment; the proposal section below never replaces these inputs.

In [2]:
data_dir = Path("../examples/evaluation")
scope = "PRIMARY"
output_dir = Path("../local/evaluation/answer_example")
manifest = json.loads((data_dir / "manifest.json").read_text(encoding="utf-8"))
metadata = manifest["metadata"]
assert metadata["dataset_kind"] in ("illustrative", "accepted")
assert scope in metadata["scopes"]
paths = verify_artifacts(data_dir, manifest)
questions = read_jsonl(paths["questions"])
reviews = [row for row in read_jsonl(paths["answer_reviews"]) if row["scope"] == scope]
saved = [row for row in read_jsonl(paths["outcomes"]) if row["scope"] == scope]
references = [row for row in read_jsonl(paths["references"]) if row["scope"] == scope]
if metadata["dataset_kind"] == "accepted":
    assert data_dir.resolve().is_relative_to(Path("../local").resolve())
    assert metadata["acceptance_ids"][scope]
print(
    {
        "input_kind": metadata["dataset_kind"],
        "scope": scope,
        "review_revision": metadata["review_revision_ids"][scope],
        "questions": len(questions),
    }
)

{'input_kind': 'illustrative', 'scope': 'PRIMARY', 'review_revision': 'EXAMPLE-REVIEW-PRIMARY', 'questions': 28}


## Ordinal rubrics and answer behaviour

Correctness uses the reference requirements, permissible variants, prohibited facts
and numeric tolerances. Faithfulness uses the complete native generation context.
Citation correctness requires directly supporting source-page evidence; an indirect
graph carrier alone cannot supply direct citation credit. Grades 0–3 are ordinal.
Answer decisions distinguish content, caveated content, abstention and technical failure.

In [3]:
rubric = [
    (
        3,
        "Complete and correct",
        "All material claims supported",
        "Direct support for all material claims",
    ),
    (2, "Minor defect", "Minor unsupported detail", "Minor citation defect"),
    (
        1,
        "Material partial answer",
        "Material claim weak or unsupported",
        "Material missing or partial citation",
    ),
    (
        0,
        "Wrong or materially incomplete; A abstention",
        "Material hallucination or contradiction",
        "Missing, invalid or contradictory support",
    ),
]
display(
    Markdown(
        tabulate(
            rubric,
            headers=["Grade", "Correctness", "Faithfulness", "Citation"],
            tablefmt="github",
        )
    )
)

|   Grade | Correctness                                  | Faithfulness                            | Citation                                  |
|---------|----------------------------------------------|-----------------------------------------|-------------------------------------------|
|       3 | Complete and correct                         | All material claims supported           | Direct support for all material claims    |
|       2 | Minor defect                                 | Minor unsupported detail                | Minor citation defect                     |
|       1 | Material partial answer                      | Material claim weak or unsupported      | Material missing or partial citation      |
|       0 | Wrong or materially incomplete; A abstention | Material hallucination or contradiction | Missing, invalid or contradictory support |

## Preserve NA and missing output

Correctness and the PRIMARY/FINAL quality comparisons use answerable questions.
An A-abstention has correctness zero; its faithfulness and citation are not
applicable. Correct behaviour applies to both A and U. A technical failure is
missing, with no valid zero and no abstention credit. The function accepts fixed
accepted assessments or explicitly illustrative values, and rejects model proposals.

In [4]:
derived = {}
for review in reviews:
    values = answer_outcomes(review)
    for metric, result in values.items():
        if review["cohort"] == "A" or metric == "correct_behavior":
            key = (review["condition_id"], review["question_id"], metric)
            assert key not in derived
            derived[key] = result
saved_by_key = {
    (row["condition_id"], row["question_id"], row["metric"]): row for row in saved
}
assert len(saved_by_key) == len(saved)
for key, result in derived.items():
    expected = saved_by_key[key]
    assert result["value"] == expected["value"]
    assert (result["status"] == "valid") == (expected["status"] == "valid")
display(
    Markdown(
        tabulate(
            [
                (key[0], key[1], key[2], value["value"], value["status"])
                for key, value in list(derived.items())[:5]
            ],
            headers=["Condition", "Question", "Outcome", "Value", "Status"],
            tablefmt="github",
        )
    )
)

| Condition   | Question      | Outcome              |   Value | Status   |
|-------------|---------------|----------------------|---------|----------|
| C0          | EXAMPLE-Q-001 | answer_correctness   |       0 | valid    |
| C0          | EXAMPLE-Q-001 | faithfulness         |       0 | valid    |
| C0          | EXAMPLE-Q-001 | citation_correctness |       3 | valid    |
| C0          | EXAMPLE-Q-001 | correct_behavior     |       1 | valid    |
| G0          | EXAMPLE-Q-001 | answer_correctness   |       1 | valid    |

## Total, applicable and valid questions

`N_total` is the selected target population, `N_app` removes known semantic NA,
and `k` counts valid values. Missing output or review reduces k. Correctness and
retrieval target A; conditional quality uses its outcome-specific applicable subset.
Ordinal results are shown as grade counts, with no mean grade.

In [5]:
target_ids = {q["question_id"] for q in questions if q["cohort"] == "A"}
coverage_rows = []
for cid in metadata["conditions_by_scope"][scope]:
    for metric in ("answer_correctness", "faithfulness", "citation_correctness"):
        rows = [
            row
            for row in saved
            if row["condition_id"] == cid
            and row["metric"] == metric
            and row["question_id"] in target_ids
        ]
        na = {
            row["question_id"]
            for row in rows
            if row["status"] in ("not_applicable_no_answer", "citation_not_required")
        }
        values = [row["value"] for row in rows if row["status"] == "valid"]
        coverage = surface_coverage(
            len(target_ids), len(target_ids) - len(na), len(values)
        )
        counts = {grade: values.count(grade) for grade in range(4)}
        coverage_rows.append(
            {"condition_id": cid, "metric": metric, **coverage, "grade_counts": counts}
        )
display(
    Markdown(
        tabulate(
            [
                (
                    row["condition_id"],
                    row["metric"],
                    row["N_total"],
                    row["N_app"],
                    row["k"],
                    row["missing"],
                    row["grade_counts"],
                )
                for row in coverage_rows[:6]
            ],
            headers=[
                "Condition",
                "Outcome",
                "N_total",
                "N_app",
                "k",
                "Missing",
                "Grade counts",
            ],
            tablefmt="github",
        )
    )
)

| Condition   | Outcome              |   N_total |   N_app |   k |   Missing | Grade counts             |
|-------------|----------------------|-----------|---------|-----|-----------|--------------------------|
| C0          | answer_correctness   |        24 |      24 |  23 |         1 | {0: 7, 1: 5, 2: 5, 3: 6} |
| C0          | faithfulness         |        24 |      23 |  22 |         1 | {0: 6, 1: 5, 2: 5, 3: 6} |
| C0          | citation_correctness |        24 |      23 |  22 |         1 | {0: 6, 1: 5, 2: 5, 3: 6} |
| G0          | answer_correctness   |        24 |      24 |  24 |         0 | {0: 6, 1: 6, 2: 6, 3: 6} |
| G0          | faithfulness         |        24 |      24 |  24 |         0 | {0: 6, 1: 6, 2: 6, 3: 6} |
| G0          | citation_correctness |        24 |      24 |  24 |         0 | {0: 6, 1: 6, 2: 6, 3: 6} |

## Form the same-question baseline contrast

Every contrast uses complete question pairs for its particular outcome. The
example compares G0 with C0. Ordinal change is a win/tie/loss sign; ties remain
in the dominance denominator. Numeric and binary change is variant minus reference.
Below 20 valid pairs the function retains counts and suppresses the aggregate.

The baseline effect retains the G0−C0 direction of Table 7. Its ordinal count order is C0 higher, tie, G0 higher.

In [6]:
cluster_by_question = connected_components(
    {row["question_id"]: set(row["question_units"]) for row in references}
)
contrasts = {}
for metric, scale in config.OUTCOME_SCALES.items():
    ids = [
        q["question_id"]
        for q in questions
        if metric == "correct_behavior" or q["cohort"] == "A"
    ]
    values, na = outcome_inputs(saved, ("C0", "G0"), metric, ids)
    contrasts[metric] = paired_intervention_change(
        values["C0"],
        values["G0"],
        question_ids=ids,
        scale=scale,
        cluster_by_question=cluster_by_question,
        non_applicable_question_ids=na,
    )
display(
    Markdown(
        tabulate(
            [
                (
                    metric,
                    row["N_total"],
                    row["N_app"],
                    row["k"],
                    (
                        row["change_counts"]["negative"],
                        row["change_counts"]["zero"],
                        row["change_counts"]["positive"],
                    ),
                    row["standard"]["estimate"] if row["standard"] else None,
                )
                for metric, row in contrasts.items()
            ],
            headers=[
                "Outcome",
                "N_total",
                "N_app",
                "k",
                "C0 higher / tie / G0 higher",
                "Paired estimate",
            ],
            tablefmt="github",
        )
    )
)

| Outcome              |   N_total |   N_app |   k | C0 higher / tie / G0 higher   |   Paired estimate |
|----------------------|-----------|---------|-----|-------------------------------|-------------------|
| ndcg_at_5            |        24 |      24 |  22 | (0, 0, 22)                    |         0.167024  |
| evidence_recall_at_5 |        24 |      24 |  22 | (0, 22, 0)                    |         0         |
| answer_correctness   |        24 |      24 |  23 | (6, 0, 17)                    |         0.478261  |
| faithfulness         |        24 |      23 |  22 | (6, 0, 16)                    |         0.454545  |
| citation_correctness |        24 |      23 |  22 | (16, 0, 6)                    |        -0.454545  |
| correct_behavior     |        28 |      28 |  27 | (1, 23, 3)                    |         0.0740741 |

## Question and evidence-component bootstrap

Questions sharing evidence belong to connected components, including transitive
connections. The paired function draws 10,000 question samples and 10,000 complete
component samples with seed 20260825. Component sampling preserves unequal
component sizes. These descriptive sensitivity calculations do not establish
independent corpus sampling or a confirmatory significance claim.

In [7]:
selected = contrasts["answer_correctness"]
display(
    Markdown(
        tabulate(
            [
                (
                    name,
                    row["k"],
                    row["resampling_units"],
                    row["estimate"],
                    row["ci_lower"],
                    row["ci_upper"],
                    row["iterations"],
                    row["seed"],
                )
                for name in ("standard", "cluster")
                if (row := selected[name]) is not None
            ],
            headers=[
                "Resampling",
                "k",
                "Units",
                "Dominance",
                "Lower",
                "Upper",
                "Draws",
                "Seed",
            ],
            tablefmt="github",
        )
    )
)

| Resampling   |   k |   Units |   Dominance |    Lower |    Upper |   Draws |     Seed |
|--------------|-----|---------|-------------|----------|----------|---------|----------|
| standard     |  23 |      23 |    0.478261 | 0.130435 | 0.826087 |   10000 | 20260825 |
| cluster      |  23 |       8 |    0.478261 | 0.272727 | 0.73913  |   10000 | 20260825 |

## Compare the system gap on four complete states

For numeric outcomes the change is `(G_variant − C_variant) − (G0 − C0)` per
question. For ordinal outcomes each G−C gap is first replaced by its sign, then
the signs are differenced. All four states must be valid on that question.
The FINAL example uses its own relevance pool when `scope = "FINAL"`.

In [8]:
c_variant = "C-FINAL" if scope == "FINAL" else "C-EMBED-LARGE-1536"
g_variant = "G-FINAL" if scope == "FINAL" else "G-EMBED-LARGE-1536"
metric = "answer_correctness"
state_ids = ("C0", "G0", c_variant, g_variant)
values, na = outcome_inputs(saved, state_ids, metric, sorted(target_ids))
gap = paired_system_gap_shift(
    *(values[cid] for cid in state_ids),
    question_ids=sorted(target_ids),
    scale="ordinal",
    cluster_by_question=cluster_by_question,
    non_applicable_question_ids=na,
)
print(
    {
        "states": state_ids,
        "k_gap": gap["k_gap"],
        "change_counts": gap["change_counts"],
        "gap_shift": gap["standard"]["estimate"] if gap["standard"] else None,
    }
)

{'states': ('C0', 'G0', 'C-EMBED-LARGE-1536', 'G-EMBED-LARGE-1536'), 'k_gap': 23, 'change_counts': {'positive': 0, 'zero': 23, 'negative': 0}, 'gap_shift': 0.0}


## Count abstention behaviour separately

U-abstention is TP; U-content is FN. A-content is TN; A-abstention is FP.
Caveated content is still content. Technical failure is excluded from the four
cells and remains visible as missing. This decision surface is separate from
correctness, faithfulness and citation grades.

In [9]:
behavior = {
    cid: behavior_counts([row for row in reviews if row["condition_id"] == cid])
    for cid in metadata["conditions_by_scope"][scope]
}
display(
    Markdown(
        tabulate(
            [
                (
                    cid,
                    row["TP"],
                    row["FN"],
                    row["FP"],
                    row["TN"],
                    row["missing"],
                    row["abstention_appropriateness"],
                )
                for cid, row in behavior.items()
            ],
            headers=["Condition", "TP", "FN", "FP", "TN", "Missing", "Appropriateness"],
            tablefmt="github",
        )
    )
)

| Condition                    |   TP |   FN |   FP |   TN |   Missing |   Appropriateness |
|------------------------------|------|------|------|------|-----------|-------------------|
| C0                           |    2 |    2 |    1 |   22 |         1 |          0.888889 |
| G0                           |    3 |    1 |    0 |   24 |         0 |          0.964286 |
| C-CHUNK-STRUCT-800-0         |    3 |    1 |    0 |   24 |         0 |          0.964286 |
| G-CHUNK-STRUCT-800-0         |    2 |    2 |    0 |   24 |         0 |          0.928571 |
| C-EMBED-LARGE-1536           |    3 |    1 |    0 |   24 |         0 |          0.964286 |
| G-EMBED-LARGE-1536           |    3 |    1 |    0 |   24 |         0 |          0.964286 |
| G-ENTITY-POLICY-PROCESS-ROLE |    2 |    2 |    0 |   24 |         0 |          0.928571 |

## Bind the recalculation output

The output retains scope, input identity, coverage and full pair memberships.
It stays in ignored `local/`. Verify input and output bindings before repeating
calculations from relocated stored artefacts.

In [10]:
assert output_dir.resolve().is_relative_to(Path("../local").resolve())
output_dir.mkdir(parents=True, exist_ok=True)
result = {
    "scope": scope,
    "dataset_kind": metadata["dataset_kind"],
    "input_manifest_sha256": hashlib.sha256(
        (data_dir / "manifest.json").read_bytes()
    ).hexdigest(),
    "coverage": coverage_rows,
    "baseline_contrasts": contrasts,
    "gap": gap,
    "behavior": behavior,
    "claim_status": "descriptive",
}
(output_dir / "answer_analysis.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print({"saved_analysis": "answer_analysis.json", "model_requests": 0})
write_analysis_manifest(
    output_dir, data_dir / "manifest.json", {"answer_analysis": "answer_analysis.json"}
)

{'saved_analysis': 'answer_analysis.json', 'model_requests': 0}


{'schema_version': 'evaluation-outputs-v1',
 'input_manifest_sha256': '0d05c8c711f992f81727527d2060f1e9cdd5af0649ae2eb8df366c7cd26d31bd',
 'artifacts': {'answer_analysis': {'path': 'answer_analysis.json',
   'sha256': 'd6bcafc8aa191c48ed71dc1ab97968d721189b8aff0303034981d9218ed9773e'}}}

## Optional Azure assessment proposals

The supplied packet contains an unchanged public RFC answer, reference requirements,
its full native context and direct citation sources. Reference-corpus pages and
generation sources have separate scopes. `RUN_AZURE_JUDGE` defaults to false.
New proposals use the configured Azure deployment and a strict JSON schema. They remain separate pending-review files; accepted grades are never changed.

In [11]:
RUN_AZURE_JUDGE = False
judge_packets_path = data_dir / "judge_packets.jsonl"
judge_dir = Path("../local/evaluation/judge_proposals")
if RUN_AZURE_JUDGE:
    assert judge_dir.resolve().is_relative_to(Path("../local").resolve())
    judge_packets = read_jsonl(judge_packets_path)
    judge_dir.mkdir(parents=True, exist_ok=True)
    judge_parameters = {
        "max_output_tokens": config.JUDGE_MAX_OUTPUT_TOKENS,
        "instructions": config.JUDGE_SYSTEM_PROMPT,
        "schema": config.JUDGE_OUTPUT_SCHEMA,
    }
    binding = {
        name: os.environ[name]
        for name in (
            "AZURE_OPENAI_ENDPOINT",
            "AZURE_OPENAI_API_VERSION",
            config.JUDGE_DEPLOYMENT_VARIABLE,
        )
    }
    judge_parameters["azure_binding_sha256"] = hashlib.sha256(
        json.dumps(binding, sort_keys=True).encode()
    ).hexdigest()
    identity = hashlib.sha256(
        json.dumps(judge_parameters, sort_keys=True).encode()
    ).hexdigest()
    from openai import AzureOpenAI

    judge_client = AzureOpenAI(
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_version=os.environ["AZURE_OPENAI_API_VERSION"],
        max_retries=config.PROVIDER_MAX_ATTEMPTS - 1,
    )

### Request and retain a proposal

Each stored proposal binds packet, prompt/schema, model settings and operational
Azure settings. Reuse requires identical bindings. Empty or incomplete responses
remain errors. Human review must resolve grades and evidence before a separate
accepted-assessment file can be used in the calculation above.

In [12]:
if RUN_AZURE_JUDGE:
    for packet in judge_packets:
        packet_sha = hashlib.sha256(
            json.dumps(packet, ensure_ascii=False, sort_keys=True).encode()
        ).hexdigest()
        proposal_path = judge_dir / f"{packet_sha}.json"
        if proposal_path.exists():
            stored = json.loads(proposal_path.read_text(encoding="utf-8"))
            assert (
                stored["configuration_sha256"] == identity
                and stored["packet_sha256"] == packet_sha
            )
            continue
        response = judge_client.responses.create(
            model=os.environ[config.JUDGE_DEPLOYMENT_VARIABLE],
            instructions=config.JUDGE_SYSTEM_PROMPT,
            input=json.dumps(packet, ensure_ascii=False),
            max_output_tokens=config.JUDGE_MAX_OUTPUT_TOKENS,
            store=False,
            text={
                "format": {
                    "type": "json_schema",
                    "name": "assessment_proposal",
                    "schema": config.JUDGE_OUTPUT_SCHEMA,
                    "strict": True,
                }
            },
        )
        assert response.status == "completed" and response.output_text
        proposal = json.loads(response.output_text)
        assert proposal["case_id"] == packet["case_id"]
        assert set(proposal["evidence_refs"]) <= {
            row["source_id"] for row in packet["sources"]
        }
        if packet["cohort"] == "U":
            assert proposal["answer_correctness"] is None
        if proposal["decision_label"] == "abstains":
            assert (
                proposal["faithfulness"] is None
                and proposal["citation_correctness"] is None
            )
            if packet["cohort"] == "A":
                assert proposal["answer_correctness"] == 0
        record = {
            "assessment_state": "proposal",
            "review_status": "pending_human_review",
            "case_id": packet["case_id"],
            "packet_sha256": packet_sha,
            "configuration_sha256": identity,
            "returned_model": response.model,
            "usage": response.usage.model_dump(mode="json") if response.usage else None,
            "proposal": proposal,
        }
        proposal_path.write_text(
            json.dumps(record, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
        )